In [1]:
import ee
import time
import math
import pandas as pd
import glob
import os

ee.Authenticate(
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project='res-neur-491811')
print('GEE initialized successfully')

GEE initialized successfully


In [2]:
YEAR         = 2021
START_DATE   = f'{YEAR}-01-01'
END_DATE     = f'{YEAR}-12-31'
N_COMPOSITES = 36
N_SAMPLES    = 5000    
CDL_CONF     = 95
SCALE        = 30
RANDOM_SEED  = 42
DRIVE_FOLDER = 'MCTNet_Data'
BANDS        = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']


arkansas = ee.Geometry.Rectangle([-92.2, 34.0, -90.7, 35.0])

california = ee.Geometry.Rectangle([-122.5, 38.5, -121.0, 39.5])


# arkansas   = states.filter(ee.Filter.eq('NAME', 'Arkansas')).geometry()

# california = states.filter(ee.Filter.eq('NAME', 'California')).geometry()
print(f'Target: {N_SAMPLES} samples per state')

Target: 5000 samples per state


visualize map

In [3]:
import folium

def add_ee_layer(self, ee_object, vis_params, name):
    map_id_dict = ee.Image(ee_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


m = folium.Map(location=[35, -105], zoom_start=4)


folium.GeoJson(data=arkansas.getInfo(), style_function=lambda x: {'color':'red'}).add_to(m)


folium.GeoJson(data=california.getInfo(), style_function=lambda x: {'color':'blue'}).add_to(m)
m


In [4]:
def mask_s2_clouds(image):
    scl   = image.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
    return (image.updateMask(clear)
                 .select(BANDS)
                 .divide(10000)
                 .copyProperties(image, ['system:time_start']))

In [5]:
def build_composite_stack(roi):
    s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(roi)
                .filterDate(START_DATE, END_DATE)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                .select(BANDS + ['SCL'])
                .map(mask_s2_clouds))

    def make_composite(i):
        i          = ee.Number(i)
        t_start    = ee.Date(START_DATE).advance(i.multiply(10), 'day')
        t_end      = t_start.advance(10, 'day')
        window     = s2_col.filterDate(t_start, t_end)

        default_composite = ee.Image.constant(0).select([0] * len(BANDS), BANDS)

        is_window_empty = window.size().eq(0)

        spectral_composite_masked = ee.Algorithms.If(
            is_window_empty,
            default_composite,
            window.median()
        )
        spectral_composite_masked = ee.Image(spectral_composite_masked)

        raw_missing_mask_co = ee.Algorithms.If(
            is_window_empty,
            ee.Image.constant(1), 
            spectral_composite_masked.mask().Not() 
        )
        suffix     = i.format('%d')
        missing_band_name = ee.String('missing_').cat(suffix)
        is_missing_band = ee.Image(raw_missing_mask_co).select([0]).rename(missing_band_name)

        final_composite = spectral_composite_masked.unmask(0)

        return final_composite.addBands(is_missing_band)

    composites = ee.List.sequence(0, N_COMPOSITES - 1).map(make_composite)
    return ee.ImageCollection(composites).toBands()

In [6]:
def get_crop_label_image(roi,crop_list):
    cdl_raw    = ee.Image(f'USDA/NASS/CDL/{YEAR}')
    crop_type  = cdl_raw.select('cropland')
    confidence = cdl_raw.select('confidence')

    worldcover    = (ee.ImageCollection('ESA/WorldCover/v200')
                       .filterBounds(roi).first())
    cropland_mask = worldcover.select('Map').eq(40)
    conf_mask     = confidence.gte(CDL_CONF)
    roi_mask      = ee.Image.constant(1).clip(roi)

    masked = (crop_type
                .updateMask(conf_mask)
                .updateMask(cropland_mask)
                .updateMask(roi_mask))

    remapped = masked.remap(
        crop_list,       
        crop_list,       
        defaultValue=99    
    )

    return remapped.updateMask(roi_mask)

In [7]:
def export_samples_batched(state_geometry, state_name, samples_per_class,crop_list,target_samples):

    stack      = build_composite_stack(state_geometry)
    crop_label = get_crop_label_image(state_geometry,crop_list)
    full_image = stack.addBands(crop_label.rename('cropland'))
    target_crops = crop_list + [99]

    samples = full_image.stratifiedSample(
            numPoints  = 0,       
            classBand  = 'cropland',    
            region     = state_geometry,
            scale      = SCALE,
            seed       = RANDOM_SEED,
            tileScale  = 4,
            geometries = False,
            classValues    = target_crops,   
            classPoints    = target_samples  
        )
    task = ee.batch.Export.table.toDrive(
            collection     = samples,
            description    = f'MCTNet_{state_name}',
            folder         = DRIVE_FOLDER,
            fileNamePrefix = f'{state_name.lower()}',
            fileFormat     = 'CSV'
        )
    task.start()

    return task


In [8]:
known_codes = [1, 2, 3, 5]        
target_samples_ar = [1522, 762, 2423, 4677,616]
ar_tasks = export_samples_batched(arkansas,   'Arkansas', samples_per_class=2000, crop_list=known_codes,target_samples=target_samples_ar)
known_codes = [3, 36, 69, 75, 204] 
target_samples_ca = [2037, 974, 2054, 783,640,3512]
ca_tasks = export_samples_batched(california, 'California', samples_per_class=1667, crop_list=known_codes,target_samples=target_samples_ca)

In [ ]:
def get_climate_covariates(roi):
    """
    import calendar

    sample_days = []
    for month in range(1, 13):
        last_day = calendar.monthrange(2021, month)[1]
        sample_days.extend([
            (month, 10),
            (month, 20),
            (month, last_day),
        ])

    bands = []

    for i, (month, day) in enumerate(sample_days):
        dekad_num   = i + 1
        target_date = f'2021-{month:02d}-{day:02d}'

        
        import datetime
        dt       = datetime.date(2021, month, day)
        next_dt  = dt + datetime.timedelta(days=1)
        next_date = next_dt.strftime('%Y-%m-%d')

        snapshot = (
            ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")  #used this instead of ERA5/DAILY cuz 1979-01-02  –  2020-07-09
            .filterDate(target_date, next_date)
            #.filterBounds(roi)
            .first()
        )

        snapshot = ee.Image(snapshot)

        bands.append(snapshot.select('temperature_2m')
                             .rename(f'temp_d{dekad_num:02d}'))
        bands.append(snapshot.select('total_precipitation_sum')
                             .rename(f'precip_d{dekad_num:02d}'))
        bands.append(snapshot.select('temperature_2m_max')
                             .rename(f'temp_max_d{dekad_num:02d}'))

    climate_stack = ee.Image.cat(bands).clip(roi)
    
    #TOPO
    elevation_ds = ee.Image("NOAA/NGDC/ETOPO1").clip(roi)
    elevation = elevation_ds.select('bedrock')
    
    landform_ds = ee.Image('CSP/ERGo/1_0/Global/ALOS_landforms')
    landform = landform_ds.select('constant');

    topography_stack = elevation.addBands(landform)
    """

    #SOIL 

    soil_ph = ee.Image("OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02").select('b0').rename('soil_ph')
    soil_oc = ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02").select('b0').rename('soil_oc')
    soil_clay = ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02").select('b0').rename('soil_clay')
    soil_stack = ee.Image.cat([soil_ph, soil_oc, soil_clay]).clip(roi)
       
    return soil_stack#.addBands(topography_stack).addBands(soil_stack)

In [32]:
def export_samples_with_covariates(state_geometry, state_name, crop_list, target_samples):
    
    stack = build_composite_stack(state_geometry)
    covariates = get_climate_covariates(state_geometry)
    crop_label = get_crop_label_image(state_geometry, crop_list)
    full_image = stack.addBands(covariates).addBands(crop_label.rename('cropland'))
    target_crops = crop_list + [99]
    
    samples = full_image.stratifiedSample(
            numPoints  = 0,
            classBand  = 'cropland',
            region     = state_geometry,
            scale      = SCALE, 
            seed       = RANDOM_SEED,
            tileScale  = 16,
            geometries = False,
            classValues = target_crops,
            classPoints = target_samples
        )
    task = ee.batch.Export.table.toDrive(
            collection = samples,
            description = f'MCTNet_Augmented_{state_name}',
            folder = DRIVE_FOLDER,
            fileNamePrefix = f'{state_name.lower()}_augmented',
            fileFormat = 'CSV'
        )
    task.start()
    return task

In [33]:
#ar_task = export_samples_with_covariates(arkansas, 'Arkansas', [1, 2, 3, 5], [1522, 762, 2423, 4677, 616])
ca_task = export_samples_with_covariates(california, 'California', [3, 36, 69, 75, 204], [2037, 974, 2054, 783,640,3512])